In [2]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore  
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from models import get_model, get_embeddings
from pdf_chunk import text_spliter   # your existing chunks

In [3]:

llm = get_model()
embedding_model = get_embeddings()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2497.79it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# ── Two splitters: parent is big, child is small ──
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

In [6]:
# ── Two stores: vectorstore for children, docstore for parents ──
vectorstore = Chroma(
    collection_name="child_chunks",
    embedding_function=embedding_model
)

docstore = InMemoryStore()   # holds parent chunks by ID

In [7]:
# ── Build the retriever ──
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,       # children go here
    docstore=docstore,             # parents go here
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,  # omit this for full-doc mode
    search_kwargs={"k": 5}         # how many child matches to fetch
)


In [8]:
# ── Index your documents (splits into parent + child automatically) ──
retriever.add_documents(text_spliter, ids=None)

In [9]:
docs = retriever.invoke("what is Rohanta's current university?")

In [11]:
for i in docs:
    print(i)

page_content='Before transitioning fully into industry, Rohanta taught undergraduate Computer Science to 100+ students. He designed curriculum and lab exercises focused on Python, Data Structures, and algorithms. He conducted workshops on data science tools including NumPy, Pandas, and Matplotlib. He supervised applied AI projects involving NLP and data-driven solutions. He guided NLP-based student research projects. This phase gave him strong communication and mentoring skills that most engineers do not' metadata={'source': 'my_data.txt'}
page_content='Rohanta is currently working as a self-employed AI/ML Consultant while simultaneously pursuing his MSc in Artificial Intelligence at the Berlin School of Business and Innovation (BSBI). In this role, he develops AI/ML solutions for clients using Python, TensorFlow, and HuggingFace. He builds LLM-powered applications and Generative AI prototypes using LangChain and LangGraph. He consults on ML architecture decisions, model optimization, 

In [12]:
# ── Full LCEL chain ──
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below.

Context: {context}
Question: {question}
""")

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print(chain.invoke("what is Rohanta's current university?"))

Rohanta is currently pursuing his MSc in Artificial Intelligence at the Berlin School of Business and Innovation (BSBI).
